# Terratorch Iterate 

This notebook will guide you through how to use [iterate](https://github.com/terrastackai/iterate)

In [ ]:
# !pip install -qq terratorch==1.2.4 optuna optuna-dashboard terratorch-iterate==0.4


## Overview
There are two way to use ***iterate classic*** and ***trial script***.

We are going to show the new ***trial script** approach, however there is pleny of documentation on the classic way

The trial script approach is the to use terratorch-iterate for hyperparameter optimization (HPO). This document explains why we use `train_trial.py` instead of the deprecated `iterate-classic` format.


## 1. How It Works

```
┌─────────────────┐
│  hpo_config.yaml│  ← Defines search space
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│    iterate      │  ← Orchestrates HPO with Optuna
│   (Optuna)      │
└────────┬────────┘
         │ For each trial:
         │ 1. Sample hyperparameters
         │ 2. Set ITERATE_PARAM_* env vars
         │ 3. Call train_trial.py
         │
         ▼
┌─────────────────┐
│ train_trial.py  │  ← Runs one experiment
│                 │
│ 1. Read params  │
│ 2. Run training │
│ 3. Extract loss │
│ 4. Write metric │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ base_config.yaml│  ← Standard terratorch config
└─────────────────┘
```


### Iterate HPO config

This file defines the hyperparameter search space for iterate

In [ ]:
with open("hpo_config.yaml") as f:
    print(f.read())

### Trial script

1. **Reads** hyperparameters from environment variables
2. **Runs** a training experiment with those hyperparameters
3. **Extracts** the validation metric from training logs
4. **Reports** the metric back to iterate/Optuna

##### Step 1: Reading Parameters (`get_params()`)
Reads all hyperparameters from environment variables.
```python
def get_params():
    """Read trial parameters from ITERATE_PARAM_* environment variables."""
    lr_str = os.environ.get("ITERATE_PARAM_LR")
    if lr_str is None:
        print("ERROR: ITERATE_PARAM_LR not set", file=sys.stderr)
        sys.exit(1)
    lr = float(lr_str)

    batch_size_str = os.environ.get("ITERATE_PARAM_BATCH_SIZE")
    if batch_size_str is None:
        print("ERROR: ITERATE_PARAM_BATCH_SIZE not set", file=sys.stderr)
        sys.exit(1)
    batch_size = int(batch_size_str)

    config = os.environ.get(
        "ITERATE_PARAM_CONFIG",
        "tm_multi_simple_s2_dem_stride_32_ccc.yaml",
    )
    .
    .
    .
    epochs = int(os.environ.get("ITERATE_PARAM_EPOCHS", "5"))
    return lr, batch_size, config_path, epochs

```

##### Step 2. Run terratorch fit with jsonargparse override for learning rate
```python
    cmd = [
        sys.executable, "-m", "terratorch",
        "fit",
        "-c", str(config_path),
        "--optimizer.init_args.lr", str(lr),
        "--data.init_args.batch_size", str(batch_size),
        "--trainer.max_epochs", str(epochs),
        "--trainer.default_root_dir", str(log_dir),
        "--trainer.logger", logger_json,
    ]
```

##### Step 3. Extract best validation loss from Lightning's auto-generated CSV
```python
    val_loss = best_val_loss(metrics_csv)
    print(f"[trial {trial_num}] best val_loss = {val_loss:.6f}")
```

##### Step 4.  Write metric to ITERATE_OUT_FILE (iterate reads this for Optuna)
```python
    if out_file:
        with open(out_file, "w") as fh:
            fh.write(f"val_loss: {val_loss}\n")
        print(f"[trial {trial_num}] metrics written to {out_file}")
```

## 2. Run Iterate


In [ ]:
import os
import sys
from pathlib import Path

# Set environment variable for notebook directory so trial script can find configs
os.environ['ITERATE_NB_DIR'] = os.getcwd()

# Paths
NB_DIR = Path(os.getcwd())
SCRIPT = NB_DIR / 'train_salt_marsh_trial.py'
HPO_YAML = NB_DIR / 'hpo_config.yaml'
DB_PATH = NB_DIR / 'iterate_salt_marsh.db'
ITERATE = Path(sys.executable).parent / 'iterate'

print(f"iterate   : {ITERATE}  (exists={ITERATE.exists()})")
print(f"script    : {SCRIPT}   (exists={SCRIPT.exists()})")
print(f"hpo yaml  : {HPO_YAML} (exists={HPO_YAML.exists()})")
print()

# Make script executable
!chmod 755 {SCRIPT}

# Run iterate HPO
!{ITERATE} \
    --script {SCRIPT} \
    --optuna-study-name sm_hpo \
    --optuna-db-path "sqlite:///{DB_PATH}" \
    --hpo-yaml {HPO_YAML} \
    --optuna-n-trials 2

## 3. Inspect Optuna Results

After the loop finishes, we load the Optuna study from the SQLite database and display the best trial.

In [ ]:
import optuna, sys, os
from pathlib import Path

NB_DIR = Path(os.getcwd())
if not (NB_DIR / "iterate_salt_marsh.db").exists():
    NB_DIR = NB_DIR / "examples" / "embeddings"
DB_PATH = NB_DIR / "iterate_salt_marsh.db"

study = optuna.load_study(
    study_name="sm_hpo",
    storage=f"sqlite:///{DB_PATH}",
)

print(f"Number of finished trials: {len(study.trials)}")
best = study.best_trial
print(f"\nBest trial:")
print(f"  val_loss   : {best.value:.6f}")
print(f"  lr         : {best.params['lr']:.2e}")
print(f"  batch_size : {best.params['batch_size']}")

In [ ]:
# Plot optimisation history and parameter importance
import matplotlib.pyplot as plt
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
)

ax1 = plot_optimization_history(study)
ax1.set_title("Optimisation History")
ax1.figure.tight_layout()
plt.show()

ax2 = plot_param_importances(study)
ax2.set_title("Parameter Importances")
ax2.figure.tight_layout()
plt.show()

In [ ]:
# Uncomment to launch the dashboard (blocks the notebook while running)
# !optuna-dashboard --host 0.0.0.0 --port 8080 sqlite:///iterate_salt_marsh.db

## 4. Re-train with Best Learning Rate

Use the best learning rate found by iterate/Optuna to run a full training with more epochs.

Because TerraTorch uses jsonargparse, we can pass the override directly on the CLI without modifying the base.yaml.

In [ ]:
import optuna, sys, os
from pathlib import Path

NB_DIR = Path(os.getcwd())
if not (NB_DIR / "iterate_salt_marsh.db").exists():
    NB_DIR = NB_DIR / "examples" / "embeddings"
DB_PATH   = NB_DIR / "iterate_salt_marsh.db"
CONFIG    = NB_DIR / "tm_multi_simple_s2_dem_stride_32_ccc.yaml"
TERRATORCH = Path(sys.executable).parent / "terratorch"

study = optuna.load_study(study_name="sm_hpo", storage=f"sqlite:///{DB_PATH}")
best_lr = study.best_trial.params["lr"]
best_batch_size = study.best_trial.params["batch_size"]
print(f"Best LR: {best_lr:.2e}")
print(f"Best batch_size: {best_batch_size}")

!{TERRATORCH} fit \
    -c {CONFIG} \
    --optimizer.init_args.lr {best_lr} \
    --data.init_args.batch_size {best_batch_size} \
    --trainer.max_epochs 2
